# Step 1 — attach data + build the labeling candidate list

Run in the **cloud workstation** before labeling. For each subject it:
1. finds + attaches Capsule 1's **classifier output** (`3D-HCR-ROI-4-class-label`), the **R1
   HCR** data it used, **and any prior `HCR-ROI-human-labeling` assets** (so already-labeled
   ROIs are skipped);
2. stages all label files into one folder and builds a **candidate list** — mostly
   keep/reject-borderline ROIs + a few confident ones — **excluding already-labeled ROIs**.

**Workflow:** run cells 1-3 → `bash code/label.sh "<sids>"` to label → to add **+100 more**,
re-run cell 3 (it re-excludes what you just labeled) and re-run `label.sh` → finally run
`create_label_asset.ipynb` to publish.

In [3]:
import os, sys
from pathlib import Path
from lamf_analysis.code_ocean import code_ocean_utils as cou
from codeocean.data_asset import DataAssetSearchParams
from codeocean.components import SearchFilter

SUBJECT_IDS = ['790322']                        # <-- edit: one id or several
N_ROIS      = 100                               # ROIs per subject per batch
CLASSIFIER_SUFFIX = 'HCR-ROI-label'  # Capsule 1's capture suffix
LABEL_SUFFIX      = 'HCR-ROI-human-labeling'    # prior human-label assets
co = cou.get_co_client()

def _prior_label_assets(sid):
    """Prior HCR-ROI-human-labeling assets for this subject (tag-based search)."""
    filters = [SearchFilter(key='tags', value=LABEL_SUFFIX), SearchFilter(key='tags', value=str(sid))]
    params = DataAssetSearchParams(offset=0, limit=1000, sort_order='desc', sort_field='created',
                                   archived=False, favorite=False, filters=filters)
    return co.data_assets.search_data_assets(params).results

In [4]:
# --- find + attach: classifier output, the HCR data it used, and prior label assets ---
to_attach = []
for sid in SUBJECT_IDS:
    df = cou.get_derived_assets_df(sid, process_name=CLASSIFIER_SUFFIX, data_name='HCR')
    if df.empty:
        print(f'[{sid}] NO classifier output ({CLASSIFIER_SUFFIX})'); continue
    row = df.sort_values(['derived_date', 'derived_time']).iloc[-1]   # latest run
    clf_id = row.derived_asset_id; to_attach.append(clf_id)
    hcr_name = None
    for iid in (co.data_assets.get_data_asset(clf_id).provenance.data_assets or []):
        a = co.data_assets.get_data_asset(iid)
        if a.name.startswith(f'HCR_{sid}'):
            to_attach.append(iid); hcr_name = a.name; break
    priors = _prior_label_assets(sid)
    to_attach += [r.id for r in priors]
    print(f'[{sid}] classifier={row.derived_asset_name}\n      HCR R1 ={hcr_name}'
          f'\n      prior label assets={len(priors)}')

to_attach = list(dict.fromkeys(to_attach))   # dedup, keep order
cou.attach_assets(to_attach)
print(f'\nattached {len(to_attach)} assets — wait until mount_state is "mounted", then run the next cell.')

[790322] classifier=HCR_790322_2025-11-05_13-00-00_processed_2025-11-10_20-36-20_HCR-ROI-label_2026-06-20_20-43-55
      HCR R1 =HCR_790322_2025-11-05_13-00-00_processed_2025-11-10_20-36-20
      prior label assets=0
asset_id: adba5f37-b5f6-4164-b727-6d46a9270b18 - mount_state: mounted
asset_id: ad1bb1fc-e054-4c42-b49f-338e08a9adf7 - mount_state: 

attached 2 assets — wait until mount_state is "mounted", then run the next cell.


### Build the candidate list (run after the assets show `mounted`; re-run for +100 more)
Stages all label files (prior assets in `/data` + this session's `/scratch/labels`) into
`/scratch/all_labels`, then writes the next-batch candidates to `/scratch/label_candidates.csv`,
**excluding everything already labeled**. Re-running after a labeling batch gives the *next* set.

In [5]:
sys.path.insert(0, '/root/capsule/code')
import importlib, make_candidates as mc; importlib.reload(mc)

DATA = Path('/root/capsule/data')
RQ   = Path('/root/capsule/scratch/roi_quality');  RQ.mkdir(parents=True, exist_ok=True)
ALL  = Path('/root/capsule/scratch/all_labels');   ALL.mkdir(parents=True, exist_ok=True)
LABELS_OUT = Path('/root/capsule/scratch/labels'); LABELS_OUT.mkdir(parents=True, exist_ok=True)

# stage classifier outputs (features + proba) into one dir for the GUI
for sid in SUBJECT_IDS:
    for kind in ('features_all', 'roi_quality_proba'):
        hits = sorted(DATA.glob(f'*/{sid}_{kind}.parquet')) or sorted(DATA.glob(f'{sid}_{kind}.parquet'))
        if hits and not (RQ / hits[0].name).exists():
            (RQ / hits[0].name).symlink_to(hits[0])

# stage all label files (prior assets in /data + this session's /scratch/labels) into one dir
label_files = []
for d in DATA.iterdir():
    if d.is_dir() and 'HCR-ROI-human-labeling' in d.name:
        label_files += sorted(d.glob('*.jsonl'))
label_files += sorted(LABELS_OUT.glob('*.jsonl'))
for f in label_files:
    dst = ALL / f'{f.parent.name}__{f.name}'
    if not dst.exists():
        try: dst.symlink_to(f)
        except OSError: import shutil; shutil.copy(f, dst)
print(f'staged {len(label_files)} prior/current label file(s) → {ALL}')

mc.make_candidates(SUBJECT_IDS, RQ, '/root/capsule/scratch/label_candidates.csv',
                   n=N_ROIS, frac_ambiguous=0.8, label_sources=[ALL])
print('\nNow run:  bash /root/capsule/code/label.sh "' + ','.join(SUBJECT_IDS) + '"')

staged 0 prior/current label file(s) → /root/capsule/scratch/all_labels
[skip] no 790322_roi_quality_proba.parquet under /root/capsule/scratch/roi_quality
wrote 0 candidates → /root/capsule/scratch/label_candidates.csv

Now run:  bash /root/capsule/code/label.sh "790322"
